In [4]:
import polars as pl
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np
import duckdb
import warnings
import pyarrow

# Suprimir os warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## Objetivo Geral

Analisar os determinantes socioeconômicos, raciais e de trajetória escolar associados ao acesso de estudantes a diferentes categorias de instituições de ensino superior no Brasil, por meio de um modelo de regressão logística multinomial estimado com os microdados do ENADE 2017.

---

## Objetivos Específicos

1. Caracterizar o perfil socioeconômico dos estudantes por categoria administrativa da IES (federal, estadual/municipal e privada)

2. Estimar a associação entre renda familiar, raça, escolaridade dos pais e tipo de escola no ensino médio com a probabilidade de o estudante estar matriculado em uma universidade pública federal

3. Verificar se políticas de ação afirmativa (cotas por renda, raça ou escola pública) modificam a relação entre baixa renda e acesso ao ensino público federal

4. Comparar os odds ratios estimados entre diferentes regiões do país e áreas de curso, identificando heterogeneidades no padrão de acesso

5. Avaliar o ajuste e o poder explicativo dos modelos estimados por meio de métricas adequadas (pseudo-R², AIC e teste de razão de verossimilhança)

In [10]:
def read_txt(file_path):
    duckdb.execute(f"""
        COPY (
            SELECT * FROM read_csv(
                '{file_path}',
                delim=';',
                header=true,
                encoding='latin-1',
                strict_mode=false,
                ignore_errors=true,
                quote='"'
            )
        )
        TO 'Enade2017.parquet'
        (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)

read_txt(r'C:\Users\Matheus\Desktop\Arquivos_PUC\Analise Descritiva e Probabilidade\Projeto final\Projeto final\MICRODADOS_ENADE_2017.txt')

In [11]:
df = pl.read_parquet('Enade2017.parquet')
df

NU_ANO,CO_IES,CO_CATEGAD,CO_ORGACAD,CO_GRUPO,CO_CURSO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO,CO_REGIAO_CURSO,NU_IDADE,TP_SEXO,ANO_FIM_EM,ANO_IN_GRAD,CO_TURNO_GRADUACAO,TP_INSCRICAO_ADM,TP_INSCRICAO,NU_ITEM_OFG,NU_ITEM_OFG_Z,NU_ITEM_OFG_X,NU_ITEM_OFG_N,NU_ITEM_OCE,NU_ITEM_OCE_Z,NU_ITEM_OCE_X,NU_ITEM_OCE_N,DS_VT_GAB_OFG_ORIG,DS_VT_GAB_OFG_FIN,DS_VT_GAB_OCE_ORIG,DS_VT_GAB_OCE_FIN,DS_VT_ESC_OFG,DS_VT_ACE_OFG,DS_VT_ESC_OCE,DS_VT_ACE_OCE,TP_PRES,TP_PR_GER,TP_PR_OB_FG,TP_PR_DI_FG,…,QE_I45,QE_I46,QE_I47,QE_I48,QE_I49,QE_I50,QE_I51,QE_I52,QE_I53,QE_I54,QE_I55,QE_I56,QE_I57,QE_I58,QE_I59,QE_I60,QE_I61,QE_I62,QE_I63,QE_I64,QE_I65,QE_I66,QE_I67,QE_I68,QE_I69,QE_I70,QE_I71,QE_I72,QE_I73,QE_I74,QE_I75,QE_I76,QE_I77,QE_I78,QE_I79,QE_I80,QE_I81
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str
2017,1,1,10028,5710,3,1,5103403,51,5,26,"""F""",2007,2012,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""DCBBCEAD""","""01111111""","""EEEADDBCABADCCAAAEEDDDCAACB""","""109900119191019090101910091""",555,555,555,555,…,2,2,1,2,2,5,5,2,3,1,3,3,4,3,3,5,3,2,2,2,1,4,4,3,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,1,1,10028,5710,3,1,5103403,51,5,23,"""F""",2013,2013,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""CCBBCEBD""","""11111101""","""ECDEAADCABADDCDEAADCDCCBECB""","""109901019191119090001911191""",555,555,555,555,…,6,6,6,5,3,6,6,6,6,3,3,5,5,6,5,6,6,4,6,4,3,6,6,6,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,1,1,10028,5710,3,1,5103403,51,5,23,"""M""",2011,2013,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""CCDBCEAD""","""11011111""","""EADEEAAACAAECDACDEEBCCECBEA""","""109911009090009190100900090""",555,555,555,333,…,4,4,1,1,3,6,6,4,4,1,3,2,1,2,2,3,2,2,2,7,1,1,1,3,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,1,1,10028,5710,3,1,5103403,51,5,23,"""M""",2011,2013,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""CABBCEBD""","""10111101""","""BEDDEABECBADDACCEACBDECAEAB""","""009911109191109190001910191""",555,555,555,555,…,5,4,4,3,4,6,5,3,6,5,5,5,4,5,5,6,4,5,5,5,5,5,5,5,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,1,1,10028,5710,3,1,5103403,51,5,24,"""M""",2010,2013,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""CCBBCDAE""","""11111010""","""EDDDDDBEAABDDDAECBBBEEEDAAB""","""119900109091109091000900091""",555,555,555,555,…,2,3,3,3,2,4,4,2,4,1,3,3,3,3,2,5,2,2,2,3,3,4,2,3,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2017,19578,2,10022,6208,5001279,1,3513009,35,3,26,"""M""",2013,2014,4,0,1,8,0,0,0,27,1,4,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDCADEDDECEBABCABZ""","""EDXXEABCXBDCADEXDECEBABCABZ""",null,null,null,null,888,888,888,888,…,4,4,3,3,4,4,4,7,7,7,4,4,4,3,4,4,3,3,3,4,3,4,4,3,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,19578,2,10022,6208,5001279,1,3513009,35,3,41,"""M""",2013,2014,4,0,1,8,0,0,0,27,1,4,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDCADEDDECEBABCABZ""","""EDXXEABCXBDCADEXDECEBABCABZ""",null,null,null,null,888,888,888,888,…,5,2,4,4,5,2,5,8,3,4,3,2,2,4,2,2,4,4,4,5,1,4,4,1,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,19578,2,10022,6208,5001279,1,3513009,35,3,22,"""M""",2013,2014,4,0,1,8,0,0,0,27,1,4,0,

In [12]:
df.head()

NU_ANO,CO_IES,CO_CATEGAD,CO_ORGACAD,CO_GRUPO,CO_CURSO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO,CO_REGIAO_CURSO,NU_IDADE,TP_SEXO,ANO_FIM_EM,ANO_IN_GRAD,CO_TURNO_GRADUACAO,TP_INSCRICAO_ADM,TP_INSCRICAO,NU_ITEM_OFG,NU_ITEM_OFG_Z,NU_ITEM_OFG_X,NU_ITEM_OFG_N,NU_ITEM_OCE,NU_ITEM_OCE_Z,NU_ITEM_OCE_X,NU_ITEM_OCE_N,DS_VT_GAB_OFG_ORIG,DS_VT_GAB_OFG_FIN,DS_VT_GAB_OCE_ORIG,DS_VT_GAB_OCE_FIN,DS_VT_ESC_OFG,DS_VT_ACE_OFG,DS_VT_ESC_OCE,DS_VT_ACE_OCE,TP_PRES,TP_PR_GER,TP_PR_OB_FG,TP_PR_DI_FG,…,QE_I45,QE_I46,QE_I47,QE_I48,QE_I49,QE_I50,QE_I51,QE_I52,QE_I53,QE_I54,QE_I55,QE_I56,QE_I57,QE_I58,QE_I59,QE_I60,QE_I61,QE_I62,QE_I63,QE_I64,QE_I65,QE_I66,QE_I67,QE_I68,QE_I69,QE_I70,QE_I71,QE_I72,QE_I73,QE_I74,QE_I75,QE_I76,QE_I77,QE_I78,QE_I79,QE_I80,QE_I81
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str
2017,1,1,10028,5710,3,1,5103403,51,5,26,"""F""",2007,2012,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""DCBBCEAD""","""01111111""","""EEEADDBCABADCCAAAEEDDDCAACB""","""109900119191019090101910091""",555,555,555,555,…,2,2,1,2,2,5,5,2,3,1,3,3,4,3,3,5,3,2,2,2,1,4,4,3,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,1,1,10028,5710,3,1,5103403,51,5,23,"""F""",2013,2013,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""CCBBCEBD""","""11111101""","""ECDEAADCABADDCDEAADCDCCBECB""","""109901019191119090001911191""",555,555,555,555,…,6,6,6,5,3,6,6,6,6,3,3,5,5,6,5,6,6,4,6,4,3,6,6,6,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,1,1,10028,5710,3,1,5103403,51,5,23,"""M""",2011,2013,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""CCDBCEAD""","""11011111""","""EADEEAAACAAECDACDEEBCCECBEA""","""109911009090009190100900090""",555,555,555,333,…,4,4,1,1,3,6,6,4,4,1,3,2,1,2,2,3,2,2,2,7,1,1,1,3,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,1,1,10028,5710,3,1,5103403,51,5,23,"""M""",2011,2013,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""CABBCEBD""","""10111101""","""BEDDEABECBADDACCEACBDECAEAB""","""009911109191109190001910191""",555,555,555,555,…,5,4,4,3,4,6,5,3,6,5,5,5,4,5,5,6,4,5,5,5,5,5,5,5,null,null,null,null,null,null,null,null,null,null,null,null,null
2017,1,1,10028,5710,3,1,5103403,51,5,24,"""M""",2010,2013,3,0,1,8,0,0,0,27,0,8,0,"""CCBBCEAD""","""CCBBCEAD""","""EDCAEABCDBDDDCBCBBEADDCBEDB""","""EDXXEABCXBXDDCXCXBEADXCBEXB""","""CCBBCDAE""","""11111010""","""EDDDDDBEAABDDDAECBBBEEEDAAB""","""119900109091109091000900091""",555,555,555,555,…,2,3,3,3,2,4,4,2,4,1,3,3,3,3,2,5,2,2,2,3,3,4,2,3,null,null,null,null,null,null,null,null,null,null,null,null,null


In [13]:
df.columns

['NU_ANO',
 'CO_IES',
 'CO_CATEGAD',
 'CO_ORGACAD',
 'CO_GRUPO',
 'CO_CURSO',
 'CO_MODALIDADE',
 'CO_MUNIC_CURSO',
 'CO_UF_CURSO',
 'CO_REGIAO_CURSO',
 'NU_IDADE',
 'TP_SEXO',
 'ANO_FIM_EM',
 'ANO_IN_GRAD',
 'CO_TURNO_GRADUACAO',
 'TP_INSCRICAO_ADM',
 'TP_INSCRICAO',
 'NU_ITEM_OFG',
 'NU_ITEM_OFG_Z',
 'NU_ITEM_OFG_X',
 'NU_ITEM_OFG_N',
 'NU_ITEM_OCE',
 'NU_ITEM_OCE_Z',
 'NU_ITEM_OCE_X',
 'NU_ITEM_OCE_N',
 'DS_VT_GAB_OFG_ORIG',
 'DS_VT_GAB_OFG_FIN',
 'DS_VT_GAB_OCE_ORIG',
 'DS_VT_GAB_OCE_FIN',
 'DS_VT_ESC_OFG',
 'DS_VT_ACE_OFG',
 'DS_VT_ESC_OCE',
 'DS_VT_ACE_OCE',
 'TP_PRES',
 'TP_PR_GER',
 'TP_PR_OB_FG',
 'TP_PR_DI_FG',
 'TP_PR_OB_CE',
 'TP_PR_DI_CE',
 'TP_SFG_D1',
 'TP_SFG_D2',
 'TP_SCE_D1',
 'TP_SCE_D2',
 'TP_SCE_D3',
 'NT_GER',
 'NT_FG',
 'NT_OBJ_FG',
 'NT_DIS_FG',
 'NT_FG_D1',
 'NT_FG_D1_PT',
 'NT_FG_D1_CT',
 'NT_FG_D2',
 'NT_FG_D2_PT',
 'NT_FG_D2_CT',
 'NT_CE',
 'NT_OBJ_CE',
 'NT_DIS_CE',
 'NT_CE_D1',
 'NT_CE_D2',
 'NT_CE_D3',
 'CO_RS_I1',
 'CO_RS_I2',
 'CO_RS_I3',
 'CO_RS_I4',
 '